In [1]:
from dotenv import load_dotenv
import os
import requests
from datetime import datetime
import pprint

load_dotenv('apis.env')
API_KEY = os.getenv('API')

#### API 1: NEOWS


In [7]:
class Asteroid:
    def __init__(self, name, diameter, velocity, miss_distance):
        self.name = name
        self.diameter = diameter
        self.velocity = velocity
        self.miss_distance = miss_distance
        self.is_hazardous = False
        self.is_sentry_object = False
        self.orbital_data = None

    def __repr__(self):
        return f"Asteroid(name={self.name}, diameter={self.diameter}, velocity={self.velocity}, miss_distance={self.miss_distance})"

In [2]:
params = {
    'start_date' : '2026-07-16',
    'end_date'   : '2026-07-16',
    'api_key'    : API_KEY
}
results = requests.get("https://api.nasa.gov/neo/rest/v1/feed", params=params)

In [3]:
# print(params)
# print(results.json()[0].keys())
data = results.json()
no_of_asteroids:int = int(data['element_count'])
print(f"no:asteroids : {no_of_asteroids}\n\n\n\n")
# pprint.pprint(data)
pprint.pprint(data['near_earth_objects'])

no:asteroids : 5




{'2026-07-16': [{'absolute_magnitude_h': 23.05,
                 'close_approach_data': [{'close_approach_date': '2026-07-16',
                                          'close_approach_date_full': '2026-Jul-16 '
                                                                      '06:48',
                                          'epoch_date_close_approach': 1784184480000,
                                          'miss_distance': {'astronomical': '0.4620301279',
                                                            'kilometers': '69118723.009667573',
                                                            'lunar': '179.7297197531',
                                                            'miles': '42948382.9352275874'},
                                          'orbiting_body': 'Earth',
                                          'relative_velocity': {'kilometers_per_hour': '95584.8664596629',
                                                           

In [5]:
todays_asteroids_data = []
for asteroid_data in data['near_earth_objects']['2026-07-16']:

    asteroid = {
        'name': asteroid_data['name'],
        'id': asteroid_data['id'],
        'diameter_min': asteroid_data['estimated_diameter']['meters']['estimated_diameter_min'],
        'diameter_max': asteroid_data['estimated_diameter']['meters']['estimated_diameter_max'],
        'is_hazardous': asteroid_data['is_potentially_hazardous_asteroid'],
        'is_sentry_object': asteroid_data['is_sentry_object']
    }
    todays_asteroids_data.append(asteroid)

print(f"todays_asteroids_data : {todays_asteroids_data}\n\n\n\n")

todays_asteroids_data : [{'name': '(2009 DB1)', 'id': '3447916', 'diameter_min': 65.2461629789, 'diameter_max': 145.8948556919, 'is_hazardous': False, 'is_sentry_object': False}, {'name': '(2009 HA21)', 'id': '3457840', 'diameter_min': 179.7028548592, 'diameter_max': 401.8277992159, 'is_hazardous': True, 'is_sentry_object': False}, {'name': '(2017 FX101)', 'id': '3772995', 'diameter_min': 22.3128464417, 'diameter_max': 49.8930414151, 'is_hazardous': False, 'is_sentry_object': False}, {'name': '(2018 XQ2)', 'id': '3836857', 'diameter_min': 6.0891262211, 'diameter_max': 13.6157001539, 'is_hazardous': False, 'is_sentry_object': True}, {'name': '(2019 NX4)', 'id': '3843209', 'diameter_min': 160.1603379786, 'diameter_max': 358.1294030194, 'is_hazardous': False, 'is_sentry_object': False}]






### API 2 SDBD database


In [6]:
SDBD_API = f"https://ssd-api.jpl.nasa.gov/sbdb.api"

def get_orbit_value(elements, target_name):
    for element in elements:
        if element.get('name') == target_name:
            value = element.get('value')
            if value is None:
                return None
            return float(value)
    return None

for asteroid_data in todays_asteroids_data:
    response = requests.get(SDBD_API, params={'spk': asteroid_data['id']})
    response.raise_for_status()  # Raise an exception for HTTP errors

    data2 = response.json()
    # pprint.pprint(data2)

    # object data
    asteroid_data['spk_id'] = int(data2['object']['spkid'])
    asteroid_data['des'] = data2['object']['des']
    asteroid_data['neo'] = data2['object']['neo']
    asteroid_data['pha'] = data2['object']['pha']
    asteroid_data['orbit_class_name'] = data2['object']['orbit_class']['name']

    # orbit data
    asteroid_data['semi-major-axis'] = get_orbit_value(data2['orbit']['elements'], 'a')
    asteroid_data['eccentricity'] = get_orbit_value(data2['orbit']['elements'], 'e')
    asteroid_data['inclination'] = get_orbit_value(data2['orbit']['elements'], 'i')
    asteroid_data['longitude_of_ascending_node'] = get_orbit_value(data2['orbit']['elements'], 'om')
    asteroid_data['argument_of_perihelion'] = get_orbit_value(data2['orbit']['elements'], 'w')
    asteroid_data['mean_anomaly'] = get_orbit_value(data2['orbit']['elements'], 'ma')
    asteroid_data['orbital_period'] = get_orbit_value(data2['orbit']['elements'], 'per')
    asteroid_data['condition_code'] = int(data2['orbit']['condition_code'])
    asteroid_data['first_observation_date'] = data2['orbit']['first_obs']
    asteroid_data['soln_date'] = data2['orbit']['soln_date']
    asteroid_data['rms'] = float(data2['orbit']['rms'])
    asteroid_data['moid'] = float(data2['orbit']['moid'])
    asteroid_data['data_arc'] = int(data2['orbit']['data_arc'])
    asteroid_data['n_obs_used'] = data2['orbit']['n_obs_used']

#printing the final data for all asteroids
pprint.pprint(todays_asteroids_data)

[{'argument_of_perihelion': 240.0,
  'condition_code': 0,
  'data_arc': 5493,
  'des': '2009 DB1',
  'diameter_max': 145.8948556919,
  'diameter_min': 65.2461629789,
  'eccentricity': 0.44,
  'first_observation_date': '2009-02-19',
  'id': '3447916',
  'inclination': 4.44,
  'is_hazardous': False,
  'is_sentry_object': False,
  'longitude_of_ascending_node': 168.0,
  'mean_anomaly': 293.0,
  'moid': 0.0387,
  'n_obs_used': 78,
  'name': '(2009 DB1)',
  'neo': True,
  'orbit_class_name': 'Apollo',
  'orbital_period': 499.0,
  'pha': False,
  'rms': 0.57,
  'semi-major-axis': 1.23,
  'soln_date': '2024-03-05 05:00:03',
  'spk_id': 50447939},
 {'argument_of_perihelion': 220.0,
  'condition_code': 1,
  'data_arc': 5109,
  'des': '2009 HA21',
  'diameter_max': 401.8277992159,
  'diameter_min': 179.7028548592,
  'eccentricity': 0.729,
  'first_observation_date': '2009-04-20',
  'id': '3457840',
  'inclination': 6.41,
  'is_hazardous': True,
  'is_sentry_object': False,
  'longitude_of_ascend

### API 3 JPL Horizons API

In [13]:
JPL_API = "https://ssd.jpl.nasa.gov/api/horizons.api"
params = {
    'format': 'json',
    'EPHEM_TYPE': 'VECTORS',
    'COMMAND': "'DES=50843232;'",
    'CENTER': '500@399',  # EARTH GEOCENTER; use @10 for Sun and @0 for solar system barycenter
    'START_TIME': "'2026-07-16 00:00'",
    'STOP_TIME': "'2026-07-16 02:00'",
    'STEP_SIZE': '120m',
    'REF_PLANE': 'ECLIPTIC',
    'REF_SYSTEM': 'ICRF',
    'OUT_UNITS': 'KM-S',
    'VEC_TABLE': '2',
    'TIME_TYPE': 'TDB',
    'TIME_DIGITS': 'SECONDS',
}

response3 = requests.get(JPL_API, params=params)
response3.raise_for_status()  # Raise an exception for HTTP errors
data3 = response3.json()
pprint.pprint(data3)

{'result': '*******************************************************************************\n'
           'JPL/HORIZONS                     (2019 NX4)                '
           '2026-Jul-18 05:21:59\n'
           'Rec #:50367856 (+COV) Soln.date: 2021-Apr-15_21:26:25       # obs: '
           '19 (17 days)\n'
           ' \n'
           'IAU76/J2000 helio. ecliptic osc. elements (au, days, deg., '
           'period=Julian yrs):\n'
           ' \n'
           '  EPOCH=  2458668.5 ! 2019-Jul-04.00 (TDB)         Residual RMS= '
           '.16687\n'
           '   EC= .7057824392238835   QR= .5160916138502888   TP= '
           '2458753.6168163745\n'
           '   OM= 263.2708099416891   W=  129.559680097845    IN= '
           '24.1402096673499\n'
           '   A= 1.754115602375639    MA= 323.8896142366024   ADIST= '
           '2.992139590900989\n'
           '   PER= 2.32325            N= .424245024           ANGMOM= '
           '.016140116\n'
           '   DAN= 1.59916         